# LeetCode 2107 — Number of Unique Flavors After Sharing K Candies bằng Python

Link đề: https://leetcode.com/problems/number-of-unique-flavors-after-sharing-k-candies/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Sliding Window

> **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def shareCandies(self, candies: List[int], k: int) -> int:
        ...
```

Lộ trình:
1. **Brute force**: thử mọi đoạn `k` kẹo đưa cho em, đếm số vị còn lại bằng `set` — `O(n²)`.
2. **Cửa sổ trượt + `Counter` phần còn lại**: số vị còn lại = số khóa có đếm `> 0` — `O(n)` (**nộp bài**).
3. **Cửa sổ trượt "bị mất"**: đếm số vị mà **mọi** viên đều nằm trong cửa sổ — `O(n)`.
4. **Mảng hiệu theo lần xuất hiện đầu/cuối**: mỗi vị bị mất ở một khoảng vị trí cửa sổ liên tiếp ⇒ cộng dồn — `O(n)`, không cần trượt.

## 1. Mục tiêu học

1. Đổi góc nhìn: **phần giữ lại** = mảng bỏ đi một đoạn liên tiếp độ dài `k`.
2. Cài cửa sổ trượt độ dài cố định với `Counter`: thêm/bớt một phần tử, cập nhật số khóa dương trong `O(1)`.
3. Nhận ra vị `v` bị mất ⇔ **mọi** lần xuất hiện của `v` nằm trong cửa sổ ⇔ `first[v]` và `last[v]` đều trong cửa sổ.
4. Dùng **mảng hiệu** để cộng đóng góp của từng vị vào một khoảng vị trí cửa sổ.
5. Xử lý biên: `n = 0`, `k = 0`, `k = n`.

## 2. Đề bài & ví dụ

`candies[i]` là vị của viên kẹo thứ `i`. Bạn phải đưa cho em gái **`k` viên liên tiếp**, phần còn lại bạn giữ. Trả về **số vị khác nhau lớn nhất** bạn có thể giữ.

**Ví dụ 1:** `[1,2,2,3,4,3]`, `k = 3` → `3` (đưa `[2,2,3]` ở chỉ số 1..3, giữ `[1,4,3]`).

**Ví dụ 2:** `[2,2,2,2,3,3]`, `k = 2` → `2`.

**Ví dụ 3:** `[2,4,5]`, `k = 0` → `3`.

**Ràng buộc:** `0 ≤ n ≤ 10⁵`, `1 ≤ candies[i] ≤ 10⁵`, `0 ≤ k ≤ n`.

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [(([1, 2, 2, 3, 4, 3], 3), 3), (([2, 2, 2, 2, 3, 3], 2), 2), (([2, 4, 5], 0), 3)]
for (c, k), want in examples:
    print(f"candies = {c}, k = {k} -> {want}")

candies = [1, 2, 2, 3, 4, 3], k = 3 -> 3
candies = [2, 2, 2, 2, 3, 3], k = 2 -> 2
candies = [2, 4, 5], k = 0 -> 3


## 3. Kiến thức nền

### 3.1 `Counter` và số khóa có đếm dương

Khi giảm đếm về `0`, nên **xóa khóa** (hoặc theo dõi riêng biến `distinct`) để `len(counter)` đúng bằng số vị còn lại.

In [2]:
cnt = Counter([1, 2, 2, 3])
cnt[2] -= 1; cnt[1] -= 1
print(cnt, "| len =", len(cnt), "| số khóa > 0 =", sum(1 for v in cnt.values() if v > 0))
cnt += Counter()          # mẹo: phép cộng với Counter rỗng loại bỏ khóa <= 0
print(cnt, "| len =", len(cnt))

Counter({2: 1, 3: 1, 1: 0}) | len = 3 | số khóa > 0 = 2
Counter({2: 1, 3: 1}) | len = 2


### 3.2 Cửa sổ trượt độ dài cố định

Cửa sổ `[l, l + k − 1]`; khi `l` tăng 1: phần tử `candies[l − 1]` **rời** cửa sổ (quay về phần giữ lại), phần tử `candies[l + k − 1]` **vào** cửa sổ (bị lấy khỏi phần giữ lại).

### 3.3 Mảng hiệu (difference array)

Muốn cộng `+1` cho mọi vị trí trong `[a, b]`: `D[a] += 1`, `D[b + 1] −= 1`, rồi cộng dồn.

In [3]:
D = [0] * 8
for a, b in [(1, 3), (2, 5), (6, 6)]:
    D[a] += 1; D[b + 1] -= 1
acc, out = 0, []
for x in D[:-1]:
    acc += x; out.append(acc)
print("độ phủ:", out)

độ phủ: [0, 1, 2, 2, 1, 1, 1]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 `[1,2,2,3,4,3]`, `k = 3` — thử mọi cửa sổ:

| cửa sổ (chỉ số) | đưa em | giữ lại | số vị giữ |
|---|---|---|---|
| [0, 2] | 1 2 2 | 3 4 3 | 2 |
| [1, 3] | 2 2 3 | 1 4 3 | **3** |
| [2, 4] | 2 3 4 | 1 2 3 | **3** |
| [3, 5] | 3 4 3 | 1 2 2 | 2 |

Đáp án `3`. **Quan sát:** vị bị mất chỉ khi **tất cả** viên của vị đó nằm trong cửa sổ. Cửa sổ `[1, 3]` lấy cả hai viên `2` nhưng chỉ một viên `3` ⇒ chỉ mất vị `2`. Tổng số vị là `4` ⇒ giữ `4 − 1 = 3`.

In [4]:
c, k = examples[0][0]
total = len(set(c))
for l in range(len(c) - k + 1):
    win = c[l:l + k]; keep = c[:l] + c[l + k:]
    lost = [v for v in set(win) if v not in keep]
    print(f"cửa sổ [{l},{l + k - 1}] = {win}: mất vị {sorted(lost)} -> giữ {total - len(lost)}")

cửa sổ [0,2] = [1, 2, 2]: mất vị [1, 2] -> giữ 2
cửa sổ [1,3] = [2, 2, 3]: mất vị [2] -> giữ 3
cửa sổ [2,4] = [2, 3, 4]: mất vị [4] -> giữ 3
cửa sổ [3,5] = [3, 4, 3]: mất vị [3, 4] -> giữ 2


## 5. Cách 1 — Brute Force / baseline

In [5]:
def share_brute(candies, k):
    n = len(candies)
    if n == 0:
        return 0
    best = 0
    for l in range(n - k + 1):
        best = max(best, len(set(candies[:l]) | set(candies[l + k:])))
    return best

for (c, k), want in examples:
    print(share_brute(c, k), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** `n − k + 1` cửa sổ × `O(n)` dựng `set` ⇒ `O(n²)` (`10¹⁰` với `n = 10⁵`). Space `O(n)`.

**Nút thắt:** hai cửa sổ liên tiếp chỉ khác nhau **hai** phần tử.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (trượt):** giữ `Counter` của **phần giữ lại**. Ban đầu (cửa sổ `[0, k−1]`): đếm `candies[k..n−1]`. Trượt `l → l + 1`: thêm `candies[l]` vào phần giữ lại, bớt `candies[l + k]`. Số vị = số khóa dương.

**Insight 2 (mất):** gọi `inside[v]` = số viên vị `v` trong cửa sổ, `total[v]` = tổng số viên vị `v`. Vị `v` bị mất ⇔ `inside[v] = total[v]`. Đáp án `= D − min_cửa sổ (số vị bị mất)` với `D` = số vị khác nhau.

**Insight 3 (khoảng):** vị `v` bị mất ở cửa sổ `[l, l + k − 1]` ⇔ `l ≤ first[v]` và `last[v] ≤ l + k − 1` ⇔ `l ∈ [last[v] − k + 1, first[v]]` (giao với `[0, n − k]`). Mỗi vị đóng góp `+1` cho một **khoảng** giá trị `l` ⇒ mảng hiệu.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cửa sổ trượt với `Counter` phần giữ lại (code nộp)

In [6]:
def share_counter(candies, k):
    n = len(candies)
    keep = Counter(candies[k:])                 # phần giữ lại khi cửa sổ là [0, k-1]
    best = len(keep)
    for l in range(1, n - k + 1):               # cửa sổ [l, l+k-1]
        keep[candies[l - 1]] += 1               # rời cửa sổ -> quay về phần giữ
        out = candies[l + k - 1]                # vào cửa sổ -> rời phần giữ
        keep[out] -= 1
        if keep[out] == 0:
            del keep[out]
        best = max(best, len(keep))
    return best

for (c, k), want in examples:
    print(share_counter(c, k), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
3 kỳ vọng 3


Với `k = 0`: vòng lặp chạy `l = 1..n` nhưng mỗi bước vừa thêm vừa bớt cùng một phần tử… cẩn thận: với `k = 0`, `candies[l + k − 1] = candies[l − 1]` — thêm rồi bớt cùng phần tử ⇒ không đổi, `best = len(keep) = D` ✓. Với `n = 0`: `Counter()` rỗng ⇒ `0` ✓.

Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** `O(n)` thời gian, `O(D)` bộ nhớ.

### 7.2 Cách 3 — Đếm số vị bị mất trong cửa sổ

In [7]:
def share_lost(candies, k):
    n = len(candies)
    if n == 0:
        return 0
    total = Counter(candies); D = len(total)
    if k == 0:
        return D
    inside = Counter(); lost = 0
    def add(v):
        nonlocal lost
        inside[v] += 1
        if inside[v] == total[v]: lost += 1
    def remove(v):
        nonlocal lost
        if inside[v] == total[v]: lost -= 1
        inside[v] -= 1
    for i in range(k): add(candies[i])
    best_lost = lost
    for l in range(1, n - k + 1):
        remove(candies[l - 1]); add(candies[l + k - 1])
        best_lost = min(best_lost, lost)
    return D - best_lost

for (c, k), want in examples:
    print(share_lost(c, k), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** `O(n)`, `O(D)`. Góc nhìn "tối thiểu hóa số vị bị mất" thường dễ suy luận hơn và mở rộng tốt (Bài 5).

### 7.3 Cách 4 — Mảng hiệu theo `first`/`last`

In [8]:
def share_difference(candies, k):
    n = len(candies)
    if n == 0:
        return 0
    first, last = {}, {}
    for i, v in enumerate(candies):
        first.setdefault(v, i); last[v] = i
    D = len(first)
    if k == 0:
        return D
    W = n - k + 1                               # số vị trí cửa sổ l = 0..n-k
    diff = [0] * (W + 1)
    for v in first:
        lo = max(0, last[v] - k + 1); hi = min(first[v], W - 1)
        if lo <= hi:                            # vị v bị mất với mọi l trong [lo, hi]
            diff[lo] += 1; diff[hi + 1] -= 1
    best_lost, acc = None, 0
    for l in range(W):
        acc += diff[l]
        best_lost = acc if best_lost is None else min(best_lost, acc)
    return D - best_lost

for (c, k), want in examples:
    print(share_difference(c, k), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ. Không cần "trượt" — mỗi vị xử lý một lần. Khoảng `[last − k + 1, first]` rỗng khi `last − first + 1 > k` (vị trải dài hơn cửa sổ — không bao giờ mất).

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1: in cửa sổ, phần tử rời/vào, `keep` và số vị.

In [9]:
def trace_counter(candies, k):
    n = len(candies); keep = Counter(candies[k:])
    print(f"cửa sổ [0,{k - 1}]: keep = {dict(keep)} -> {len(keep)} vị")
    best = len(keep)
    for l in range(1, n - k + 1):
        back = candies[l - 1]; out = candies[l + k - 1]
        keep[back] += 1; keep[out] -= 1
        if keep[out] == 0: del keep[out]
        best = max(best, len(keep))
        print(f"cửa sổ [{l},{l + k - 1}]: +{back} (về phần giữ), -{out} (vào cửa sổ): keep = {dict(keep)} -> {len(keep)} vị | best = {best}")
    return best

trace_counter([1, 2, 2, 3, 4, 3], 3)

cửa sổ [0,2]: keep = {3: 2, 4: 1} -> 2 vị
cửa sổ [1,3]: +1 (về phần giữ), -3 (vào cửa sổ): keep = {3: 1, 4: 1, 1: 1} -> 3 vị | best = 3
cửa sổ [2,4]: +2 (về phần giữ), -4 (vào cửa sổ): keep = {3: 1, 1: 1, 2: 1} -> 3 vị | best = 3
cửa sổ [3,5]: +2 (về phần giữ), -3 (vào cửa sổ): keep = {1: 1, 2: 2} -> 2 vị | best = 3


3

In [10]:
def trace_difference(candies, k):
    n = len(candies); first, last = {}, {}
    for i, v in enumerate(candies): first.setdefault(v, i); last[v] = i
    W = n - k + 1
    for v in sorted(first):
        lo = max(0, last[v] - k + 1); hi = min(first[v], W - 1)
        print(f"vị {v}: first = {first[v]}, last = {last[v]} -> mất khi l ∈ [{lo}, {hi}]" + ("" if lo <= hi else " (rỗng: không bao giờ mất)"))

trace_difference([1, 2, 2, 3, 4, 3], 3)

vị 1: first = 0, last = 0 -> mất khi l ∈ [0, 0]
vị 2: first = 1, last = 2 -> mất khi l ∈ [0, 1]
vị 3: first = 3, last = 5 -> mất khi l ∈ [3, 3]
vị 4: first = 4, last = 4 -> mất khi l ∈ [2, 3]


Đọc trace: vị `1` mất khi `l = 0`; vị `2` mất khi `l ∈ [0, 1]`; vị `3` mất khi `l = 3`; vị `4` mất khi `l ∈ [2, 3]`. Số vị mất theo `l = 0, 1, 2, 3`: `2, 1, 1, 2` ⇒ tối thiểu `1` ⇒ giữ `4 − 1 = 3` ✓.

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def shareCandies(self, candies: List[int], k: int) -> int:
        n = len(candies)
        keep = Counter(candies[k:])
        ans = len(keep)
        for l in range(1, n - k + 1):
            keep[candies[l - 1]] += 1
            out = candies[l + k - 1]
            keep[out] -= 1
            if keep[out] == 0:
                del keep[out]
            ans = max(ans, len(keep))
        return ans

sol = Solution()
for (c, k), want in examples:
    got = sol.shareCandies(c, k)
    print(got, "OK" if got == want else "SAI")

3 OK
2 OK
3 OK


## 10. Vì sao thuật toán đúng

**Cách 2 (invariant).** Trước khi xét cửa sổ `[l, l+k−1]`, `keep` là `Counter` của `candies` **ngoài** cửa sổ, và mọi khóa đều có đếm dương.
- Khởi tạo `l = 0`: `keep = Counter(candies[k:])` ✓.
- Chuyển `l − 1 → l`: phần ngoài cửa sổ được thêm `candies[l−1]` và mất `candies[l+k−1]` — đúng hai thao tác; xóa khóa khi về `0` giữ "mọi khóa dương". (Với `k = 0` hai thao tác triệt tiêu.)
⇒ `len(keep)` = số vị giữ lại của cửa sổ đó; lấy max trên mọi cửa sổ. ✓

**Cách 3.** `lost` = `|{v : inside[v] = total[v] > 0}|`, cập nhật đúng khi thêm/bớt một viên. Số vị giữ = `D − lost`. ✓

**Cách 4.** Vị `v` mất ở cửa sổ `l` ⇔ `l ≤ first[v] ≤ last[v] ≤ l + k − 1` ⇔ `last[v] − k + 1 ≤ l ≤ first[v]`. Mảng hiệu cộng đúng số vị mất tại mỗi `l`. ✓

## 11. Phân tích Time / Space

`n ≤ 10⁵`, `D` = số vị khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(n)` |
| 2. Counter phần giữ | `O(n)` | `O(D)` |
| 3. Đếm vị bị mất | `O(n)` | `O(D)` |
| 4. Mảng hiệu | `O(n)` | `O(n)` |

In [12]:
rng = random.Random(1)
tests = [("ngẫu nhiên 1..1e5", [rng.randint(1, 10**5) for _ in range(10**5)], 30000),
         ("ít vị (1..50)", [rng.randint(1, 50) for _ in range(10**5)], 500),
         ("khối liên tiếp", [i // 100 + 1 for i in range(10**5)], 250)]
for name, c, k in tests:
    res = []
    for fname, fn in [("counter", share_counter), ("lost", share_lost), ("difference", share_difference), ("submit", Solution().shareCandies)]:
        t0 = time.perf_counter(); r = fn(c, k); res.append(r)
        print(f"{name:18s} {fname:10s}: {r:>6}  {(time.perf_counter() - t0) * 1000:.1f} ms")
    assert len(set(res)) == 1

ngẫu nhiên 1..1e5  counter   :  50535  35.5 ms
ngẫu nhiên 1..1e5  lost      :  50535  48.5 ms


ngẫu nhiên 1..1e5  difference:  50535  30.8 ms


ngẫu nhiên 1..1e5  submit    :  50535  35.5 ms


ít vị (1..50)      counter   :     50  32.8 ms


ít vị (1..50)      lost      :     50  32.7 ms
ít vị (1..50)      difference:     50  8.5 ms


ít vị (1..50)      submit    :     50  24.6 ms
khối liên tiếp     counter   :    999  43.7 ms


khối liên tiếp     lost      :    999  35.2 ms
khối liên tiếp     difference:    999  8.7 ms
khối liên tiếp     submit    :    999  26.6 ms


## 12. Bộ test

In [13]:
fns = [share_counter, share_lost, share_difference, Solution().shareCandies]

for (c, k), want in examples:
    assert share_brute(c, k) == want
    for fn in fns:
        assert fn(c, k) == want

edge = [(([], 0), 0), (([7], 0), 1), (([7], 1), 0), (([1, 2, 3], 3), 0), (([1, 1, 1], 2), 1),
        (([1, 2, 1], 1), 2), (([1, 2, 3, 1], 2), 2), (([5, 5, 6, 6, 7, 7], 2), 3)]
for (c, k), want in edge:
    assert share_brute(c, k) == want, (c, k)
    for fn in fns:
        assert fn(c, k) == want, (fn, c, k)

rng = random.Random(2024)
for _ in range(2000):
    n = rng.randint(0, 20)
    c = [rng.randint(1, rng.choice([2, 4, 10])) for _ in range(n)]
    k = rng.randint(0, n)
    want = share_brute(c, k)
    for fn in fns:
        assert fn(c, k) == want, (fn, c, k)
print("ví dụ + edge + 2000 bộ ngẫu nhiên khớp brute force")

ví dụ + edge + 2000 bộ ngẫu nhiên khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không xóa khóa khi đếm về 0 (dùng `len(counter)` sai)

In [14]:
def wrong_no_delete(candies, k):
    n = len(candies); keep = Counter(candies[k:]); best = len(keep)
    for l in range(1, n - k + 1):
        keep[candies[l - 1]] += 1; keep[candies[l + k - 1]] -= 1      # SAI: khóa đếm 0 vẫn còn
        best = max(best, len(keep))
    return best

for (c, k), want in examples + [(([1, 2, 3], 3), 0)]:
    print(f"{c}, k = {k}: sai = {wrong_no_delete(c, k)} | đúng = {want}")

[1, 2, 2, 3, 4, 3], k = 3: sai = 4 | đúng = 3
[2, 2, 2, 2, 3, 3], k = 2: sai = 2 | đúng = 2
[2, 4, 5], k = 0: sai = 3 | đúng = 3
[1, 2, 3], k = 3: sai = 0 | đúng = 0


`Counter` giữ khóa có giá trị `0` ⇒ `len` đếm cả vị đã hết. **Sửa:** `del keep[x]` khi về `0`, hoặc giữ biến `distinct`.

### Lỗi 2 — Đếm số vị **trong cửa sổ** thay vì phần giữ lại

In [15]:
def wrong_minimize_window_distinct(candies, k):
    n = len(candies)
    if n == 0: return 0
    D = len(set(candies))
    return D - min(len(set(candies[l:l + k])) for l in range(n - k + 1))   # SAI: vị trong cửa sổ có thể vẫn còn ngoài

for (c, k), want in examples:
    print(f"{c}, k = {k}: sai = {wrong_minimize_window_distinct(c, k)} | đúng = {want}")

[1, 2, 2, 3, 4, 3], k = 3: sai = 2 | đúng = 3
[2, 2, 2, 2, 3, 3], k = 2: sai = 1 | đúng = 2
[2, 4, 5], k = 0: sai = 3 | đúng = 3


Một vị có mặt trong cửa sổ **không** nhất thiết bị mất — nếu còn viên khác ở ngoài. Ví dụ 1, cửa sổ `[2,2,3]` có 2 vị nhưng chỉ mất 1. **Sửa:** đếm vị bị mất **hoàn toàn** (Cách 3) hoặc đếm phần giữ lại (Cách 2).

### Lỗi 3 — Lệch chỉ số khi trượt (bớt `candies[l + k]`)

In [16]:
def wrong_shift(candies, k):
    n = len(candies); keep = Counter(candies[k:]); best = len(keep)
    for l in range(1, n - k + 1):
        keep[candies[l - 1]] += 1
        out = candies[l + k] if l + k < n else None                     # SAI: phần tử vào cửa sổ là l + k - 1
        if out is not None:
            keep[out] -= 1
            if keep[out] == 0: del keep[out]
        best = max(best, len(keep))
    return best

for (c, k), want in examples + [(([1, 2, 3, 1], 2), 2)]:
    print(f"{c}, k = {k}: sai = {wrong_shift(c, k)} | đúng = {want}")

[1, 2, 2, 3, 4, 3], k = 3: sai = 3 | đúng = 3
[2, 2, 2, 2, 3, 3], k = 2: sai = 2 | đúng = 2
[2, 4, 5], k = 0: sai = 3 | đúng = 3
[1, 2, 3, 1], k = 2: sai = 3 | đúng = 2


Cửa sổ mới `[l, l + k − 1]` nhận thêm phần tử ở chỉ số `l + k − 1`. **Sửa:** viết rõ biên cửa sổ trước khi code.

### Lỗi 4 — Mảng hiệu: quên giới hạn vị trí cửa sổ `l ≤ n − k`

In [17]:
def wrong_no_cap(candies, k):
    n = len(candies)
    if n == 0: return 0
    first, last = {}, {}
    for i, v in enumerate(candies): first.setdefault(v, i); last[v] = i
    D = len(first)
    if k == 0: return D
    diff = [0] * (n + 1)
    for v in first:
        lo = max(0, last[v] - k + 1); hi = first[v]                  # SAI: không chặn hi <= n - k
        if lo <= hi: diff[lo] += 1; diff[hi + 1] -= 1
    acc, best = 0, None
    for l in range(n):                                              # SAI: duyệt cả l không phải cửa sổ hợp lệ
        acc += diff[l]; best = acc if best is None else min(best, acc)
    return D - best

rng = random.Random(4); shown = 0
for _ in range(5000):
    c = [rng.randint(1, 4) for _ in range(rng.randint(1, 8))]; k = rng.randint(1, len(c))
    if wrong_no_cap(c, k) != share_brute(c, k):
        print(f"phản ví dụ: {c}, k = {k} | sai = {wrong_no_cap(c, k)} | đúng = {share_brute(c, k)}"); shown += 1
        if shown == 2: break

phản ví dụ: [3, 1, 4, 4], k = 2 | sai = 3 | đúng = 2
phản ví dụ: [1, 1], k = 2 | sai = 1 | đúng = 0


Vị trí `l > n − k` không phải cửa sổ hợp lệ (cửa sổ sẽ vượt cuối mảng). Ở đó gần như không vị nào "bị mất" theo công thức nên `min` bằng `0` một cách giả tạo. **Sửa:** chỉ xét `l ∈ [0, n − k]` và chặn `hi ≤ n − k`.

### Lỗi 5 — Brute force bằng cách xóa phần tử khỏi list

`c2 = c[:]; del c2[l:l+k]` mỗi cửa sổ là `O(n)` — đúng nhưng `O(n²)`; dùng trong test thì được, không dùng để nộp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Bỏ đi đúng một đoạn liên tiếp độ dài `k`" ⇒ cửa sổ trượt trên **phần bị bỏ**, theo dõi **phần còn lại**.
- Đếm số khóa phân biệt với `Counter`: luôn xóa khóa khi về `0`.
- "Vị mất ⇔ mọi lần xuất hiện nằm trong đoạn" ⇒ chỉ cần `first`/`last`.
- Họ bài: 1423 *Maximum Points You Can Obtain from Cards*, 1658 *Minimum Operations to Reduce X to Zero*, 2461 *Maximum Sum of Distinct Subarrays With Length K*, 1838 *Frequency of the Most Frequent Element*.

## 15. Biến thể của bài

1. **Tối thiểu** số vị giữ lại (bạn muốn đưa nhiều vị nhất cho em): đổi `max` thành `min`.
2. **Đưa `k` viên tùy ý (không cần liên tiếp)**: tham lam — đưa viên của các vị có **nhiều** viên trước (mất vị chỉ khi hết).
3. **Đưa hai đoạn liên tiếp** độ dài `k₁`, `k₂` rời nhau: duyệt đoạn đầu, đoạn sau dùng kết quả tiền tính — phức tạp hơn nhiều.
4. **Đếm số cửa sổ tối ưu**: Cách 4 cho số vị mất tại mọi `l` — đếm số `l` đạt min.

In [18]:
def share_any_k(candies, k):
    """Biến thể 2: đưa k viên bất kỳ; giữ tối đa bao nhiêu vị?"""
    cnt = sorted(Counter(candies).values(), reverse=True); D = len(cnt)
    spare = sum(c - 1 for c in cnt)            # số viên "dư" có thể đưa mà không mất vị
    if k <= spare: return D
    return max(0, D - (k - spare))             # mỗi viên thêm làm mất đúng một vị

def share_any_k_brute(candies, k):
    from itertools import combinations
    n = len(candies)
    return max((len(set(candies[i] for i in range(n) if i not in S)) for S in map(set, combinations(range(n), k))), default=0)

def count_optimal_windows(candies, k):
    """Biến thể 4."""
    n = len(candies); first, last = {}, {}
    for i, v in enumerate(candies): first.setdefault(v, i); last[v] = i
    W = n - k + 1; diff = [0] * (W + 1)
    for v in first:
        lo = max(0, last[v] - k + 1); hi = min(first[v], W - 1)
        if lo <= hi: diff[lo] += 1; diff[hi + 1] -= 1
    acc, lost = 0, []
    for l in range(W): acc += diff[l]; lost.append(acc)
    return lost.count(min(lost))

rng = random.Random(15)
for _ in range(300):
    c = [rng.randint(1, 4) for _ in range(rng.randint(0, 8))]; k = rng.randint(0, len(c))
    assert share_any_k(c, k) == share_any_k_brute(c, k)
print("biến thể đưa bất kỳ:", share_any_k([1, 2, 2, 3, 4, 3], 3), "| số cửa sổ tối ưu ví dụ 1:", count_optimal_windows([1, 2, 2, 3, 4, 3], 3))

biến thể đưa bất kỳ: 3 | số cửa sổ tối ưu ví dụ 1: 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | `set` mỗi cửa sổ | `O(n²)` | `O(n)` | kiểm chứng |
| 2. Counter phần giữ | trượt | `O(n)` | `O(D)` | **nộp bài** |
| 3. Đếm vị mất | trượt "trong cửa sổ" | `O(n)` | `O(D)` | suy luận theo vị mất |
| 4. Mảng hiệu | first/last | `O(n)` | `O(n)` | cần số vị mất mọi cửa sổ |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `[1, 1, 2, 2]`, `k = 2`? `[1, 2, 3, 4]`, `k = 1`? `[3, 3, 3]`, `k = 3`?

**Bài 2 (tính tay):** `[4, 1, 4, 2, 2, 5, 1]`, `k = 3`: với mỗi vị, tính `first`, `last` và khoảng `l` làm mất vị; suy ra đáp án.

**Bài 3 (điền TODO):** hoàn thiện `first_last(candies)` trả về hai dict `first`, `last`.

**Bài 4 (tự viết lại):** viết `best_window(candies, k)` trả về chỉ số `l` **nhỏ nhất** của một cửa sổ tối ưu (hoặc `0` nếu `n = 0`).

**Bài 5 (mini challenge):** mỗi vị có **trọng số** `w[v]` (bạn thích vị đó bao nhiêu); tối đa **tổng trọng số** các vị giữ lại. Viết `O(n)` bằng mảng hiệu có trọng số. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [19]:
def first_last(candies):
    # TODO: first.setdefault(v, i); last[v] = i
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[1,1,2,2]`, `k = 2`: đưa `[1,2]` ở giữa giữ được cả hai vị ⇒ `2`. (2) `[1,2,3,4]`, `k = 1`: mất đúng một vị ⇒ `3`. (3) `[3,3,3]`, `k = 3` ⇒ `0`.

**Bài 2:** (1) `n = 7`, `l ∈ [0, 4]`. (2) Vị `4`: first 0, last 2 ⇒ mất khi `l ∈ [0, 0]`; vị `1`: 1, 6 ⇒ `[4, 1]` rỗng; vị `2`: 3, 4 ⇒ `[2, 3]`; vị `5`: 5 ⇒ `[3, 4]`. (3) Số vị mất theo `l = 0..4`: `1, 0, 1, 2, 1` ⇒ min `0` ⇒ giữ `4`.

**Bài 3:** (1) Hai dict rỗng. (2) Một vòng `enumerate`. (3) Trả `(first, last)`.

**Bài 4:** (1) Dùng mảng số vị mất như Cách 4. (2) `lost.index(min(lost))`. (3) Kiểm tra bằng brute force.

**Bài 5:** (1) Mảng hiệu cộng `w[v]` thay vì `1`. (2) Đáp án `Σ w − min(trọng số mất)`. (3) Brute: thử mọi cửa sổ, cộng trọng số vị còn lại.

## 19. Đáp án tham khảo

In [20]:
# Bài 1
assert [Solution().shareCandies(c, k) for c, k in [([1, 1, 2, 2], 2), ([1, 2, 3, 4], 1), ([3, 3, 3], 3)]] == [2, 3, 0]
# Bài 2
trace_difference([4, 1, 4, 2, 2, 5, 1], 3)
assert Solution().shareCandies([4, 1, 4, 2, 2, 5, 1], 3) == share_brute([4, 1, 4, 2, 2, 5, 1], 3) == 4

# Bài 3
def first_last_ans(candies):
    first, last = {}, {}
    for i, v in enumerate(candies):
        first.setdefault(v, i); last[v] = i
    return first, last

# Bài 4
def best_window_ans(candies, k):
    n = len(candies)
    if n == 0: return 0
    first, last = first_last_ans(candies); W = n - k + 1; diff = [0] * (W + 1)
    for v in first:
        lo = max(0, last[v] - k + 1); hi = min(first[v], W - 1)
        if lo <= hi: diff[lo] += 1; diff[hi + 1] -= 1
    acc, lost = 0, []
    for l in range(W): acc += diff[l]; lost.append(acc)
    return lost.index(min(lost))

rng = random.Random(41)
for _ in range(500):
    c = [rng.randint(1, 5) for _ in range(rng.randint(1, 15))]; k = rng.randint(0, len(c))
    f, l_ = first_last_ans(c)
    assert all(c[f[v]] == v and v not in c[:f[v]] and v not in c[l_[v] + 1:] for v in f)
    l0 = best_window_ans(c, k)
    vals = [len(set(c[:l]) | set(c[l + k:])) for l in range(len(c) - k + 1)]
    assert vals[l0] == max(vals) and l0 == vals.index(max(vals))
print("Bài 4 ví dụ 1:", best_window_ans([1, 2, 2, 3, 4, 3], 3))
print("Bài 1-4: OK")

vị 1: first = 1, last = 6 -> mất khi l ∈ [4, 1] (rỗng: không bao giờ mất)
vị 2: first = 3, last = 4 -> mất khi l ∈ [2, 3]
vị 4: first = 0, last = 2 -> mất khi l ∈ [0, 0]
vị 5: first = 5, last = 5 -> mất khi l ∈ [3, 4]
Bài 4 ví dụ 1: 1
Bài 1-4: OK


In [21]:
# Bài 5 — trọng số theo vị
def share_weighted(candies, k, w):
    n = len(candies)
    if n == 0: return 0
    first, last = first_last_ans(candies); total = sum(w[v] for v in first)
    W = n - k + 1; diff = [0] * (W + 1)
    for v in first:
        lo = max(0, last[v] - k + 1); hi = min(first[v], W - 1)
        if lo <= hi: diff[lo] += w[v]; diff[hi + 1] -= w[v]
    acc, best = 0, None
    for l in range(W):
        acc += diff[l]; best = acc if best is None else min(best, acc)
    return total - best

def share_weighted_brute(candies, k, w):
    n = len(candies)
    if n == 0: return 0
    return max(sum(w[v] for v in set(candies[:l]) | set(candies[l + k:])) for l in range(n - k + 1))

rng = random.Random(55)
for _ in range(1000):
    c = [rng.randint(1, 5) for _ in range(rng.randint(0, 15))]; k = rng.randint(0, len(c))
    w = {v: rng.randint(1, 10) for v in range(1, 6)}
    assert share_weighted(c, k, w) == share_weighted_brute(c, k, w)
print("Bài 5 ví dụ 1 với w = {1:1, 2:10, 3:1, 4:1}:", share_weighted([1, 2, 2, 3, 4, 3], 3, {1: 1, 2: 10, 3: 1, 4: 1}))

Bài 5 ví dụ 1 với w = {1:1, 2:10, 3:1, 4:1}: 12


## 20. Template nhớ nhanh + Checklist

```python
# Bỏ đúng một đoạn k phần tử liên tiếp, tối đa số giá trị phân biệt còn lại
keep = Counter(a[k:])                 # cửa sổ [0, k-1]
best = len(keep)
for l in range(1, n - k + 1):         # cửa sổ [l, l+k-1]
    keep[a[l - 1]] += 1               # rời cửa sổ
    x = a[l + k - 1]                  # vào cửa sổ
    keep[x] -= 1
    if keep[x] == 0: del keep[x]
    best = max(best, len(keep))
```

**Checklist trước khi submit:**
- [ ] Xóa khóa khi đếm về `0` (hoặc biến `distinct`).
- [ ] Phần tử vào cửa sổ là `a[l + k − 1]`, rời là `a[l − 1]`.
- [ ] `k = 0` ⇒ số vị phân biệt; `n = 0` ⇒ `0`; `k = n` ⇒ `0`.
- [ ] Theo dõi **phần giữ lại**, không phải số vị trong cửa sổ.